# Module 3 Lab: Titanic Preprocessing Pipeline

**Practical Machine Learning Foundations**

**Purpose:** turn the raw Titanic dataset into model-ready features by imputing missing values, handling outliers, encoding categoricals, and scaling numerics, then assemble every step into one scikit-learn Pipeline and inspect what it produced.

**Date:** 2026-08-24 | **Author:** Nick Garner

In the preceding lessons we explored data and diagnosed its problems. Now we fix them properly. In exploration you can point pandas at the whole dataset and nothing bad happens, because you're only looking. Feature engineering for a model is stricter. Every transformation gets fit on training data only, then applied as-is to data the model has never seen. Skip that and you leak, and leakage looks like a great model right up until it hits production.

### What you will build

| Section | Focus |
|---|---|
| 1. Profile the data | Missingness, distributions, semantic types |
| 2. Impute | Mean, median, mode, domain-driven, and when not to impute at all |
| 3. Handle outliers | Z-score, IQR, Winsorization, Isolation Forest |
| 4. Encode | Label, ordinal, one-hot, target, and frequency encoding |
| 5. Scale | StandardScaler, MinMaxScaler, RobustScaler, and when each matters |
| 6. Assemble the pipeline | ColumnTransformer plus Pipeline, cross-validated and saved |
| 7. Inspect the output | Feature names, distributions, and sanity checks |

### The dataset

The Titanic passenger manifest: 891 passengers, who they were, what they paid, and whether they survived. It is the canonical teaching dataset for preprocessing because every problem we need is already in it, none of it manufactured: a column that is 77% empty, an age column with a fifth of its values missing for reasons that are not random, a fare column with a wildly skewed distribution and a genuine extreme outlier, categoricals of every flavor from binary to 681 distinct values, and a clean binary target.


## Section 0: Setup

Fixed seed for reproducibility, relative paths, and versions printed so anyone reproducing this knows what it ran on.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print("pandas", pd.__version__, "| NumPy", np.__version__,
      "| scikit-learn", sklearn.__version__)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

for folder in ["data/raw", "data/processed", "models", "outputs"]:
    os.makedirs(folder, exist_ok=True)


### Loading the data

Loading a CSV straight from a URL. The fallback below keeps the notebook runnable if the network is unavailable; in Colab the real file loads.


In [ ]:
URL = ("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

try:
    df = pd.read_csv(URL)
    print("Loaded the real Titanic dataset from the web.")
except Exception as e:
    print("Network unavailable (" + type(e).__name__ + "), building a stand-in.")
    n = 891
    pclass = rng.choice([1, 2, 3], n, p=[.24, .21, .55])
    sex = rng.choice(["male", "female"], n, p=[.65, .35])
    age = np.where(pclass == 1, rng.normal(38, 14, n), rng.normal(27, 13, n))
    age = np.clip(age, 0.5, 80).round(1)
    age[rng.choice(n, 177, replace=False)] = np.nan
    fare = np.where(pclass == 1, rng.lognormal(4.2, .7, n),
                    rng.lognormal(2.5, .6, n)).round(4)
    fare[rng.choice(n, 3, replace=False)] = 512.3292
    emb = rng.choice(["S", "C", "Q"], n, p=[.72, .19, .09]).astype(object)
    emb[rng.choice(n, 2, replace=False)] = None
    titles = rng.choice(["Mr.", "Mrs.", "Miss.", "Master.", "Dr."], n,
                        p=[.58, .14, .21, .05, .02])
    df = pd.DataFrame({
        "PassengerId": np.arange(1, n + 1),
        "Survived": rng.binomial(1, np.where(sex == "female", .74, .19)),
        "Pclass": pclass,
        "Name": ["Doe, " + t + " Passenger " + str(i) for i, t in enumerate(titles)],
        "Sex": sex, "Age": age,
        "SibSp": rng.choice([0, 1, 2, 3], n, p=[.68, .23, .06, .03]),
        "Parch": rng.choice([0, 1, 2], n, p=[.76, .13, .11]),
        "Ticket": ["T" + str(x) for x in rng.integers(1000, 9999, n)],
        "Fare": fare,
        "Cabin": np.where(rng.random(n) < 0.23,
                          rng.choice(["C85", "B42", "E12", "D33"], n), None),
        "Embarked": emb})

df.to_csv("data/raw/titanic.csv", index=False)
print("Shape:", df.shape)
df.head()


---
# Section 1: Profile the data

Never impute, cap, encode, or scale anything before you understand what you have. Profiling is three commands and five minutes, and it determines every decision that follows.


In [ ]:
print(df.dtypes)
print()
print("Unique values per column:")
print(df.nunique().sort_values())


In [ ]:
df.describe().round(2)


### The mean versus median diagnostic

The quickest read on any numeric column: compare its mean to its median (the 50% row). Close together means the distribution is roughly symmetric and mean imputation is defensible. Far apart means it is skewed and the mean is being dragged somewhere no real passenger lives.


In [ ]:
for col in ["Age", "Fare"]:
    m, med = df[col].mean(), df[col].median()
    verdict = "roughly symmetric" if abs(m - med) / med < 0.15 else "SKEWED"
    print(col.ljust(5), "mean:", round(m, 2), "| median:", round(med, 2),
          "|", verdict)

# Create a figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Age distribution
df["Age"].plot(kind="hist", ax=axes[0], bins=30, color="royalblue", edgecolor="black")
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Frequency")
axes[0].grid(axis="y", linestyle="--", alpha=0.7)

# Plot Fare distribution
df["Fare"].plot(
    kind="hist", ax=axes[1], bins=30, color="seagreen", edgecolor="black"
)
axes[1].set_title("Fare Distribution")
axes[1].set_xlabel("Fare")
axes[1].set_ylabel("Frequency")
axes[1].grid(axis="y", linestyle="--", alpha=0.7)


`Age` is close to symmetric, so its mean is a reasonable estimate of a typical passenger. `Fare` is badly skewed: the mean is more than double the median, pulled upward by a handful of enormous first class tickets. The rule of thumb is: **median for skewed, mean for symmetric**.

### Missingness

Count and severity, because 0.2% missing and 77% missing are entirely different problems with entirely different answers.

In [ ]:
miss = pd.DataFrame({
    "missing": df.isnull().sum(),
    "percent": (df.isnull().mean() * 100).round(1)
})
miss[miss["missing"] > 0]


Three columns, three severities, and as we will see, three completely different treatments:

- **Cabin, 77% missing**: past the point where imputation is honest
- **Age, 19.9% missing**: substantial, worth the effort, and the interesting case
- **Embarked, 0.2% missing**: two rows, easily handled

### Semantic types

As covered earlier, how a column is *stored* is not what it *means*. Sorting the columns by meaning now tells us which preprocessing each one needs later:

- **Continuous**: `Age`, `Fare`. These get imputed, possibly capped, and scaled.
- **Ordinal**: `Pclass`. First class outranks second outranks third, and the integers already encode that order correctly, so no encoding work is needed.
- **Nominal, low cardinality**: `Sex` (2 values), `Embarked` (3). One-hot encoding territory.
- **Nominal, high cardinality**: `Ticket` (681 distinct values), `Cabin` (147). One-hot would be ridiculous here; this is where target and frequency encoding earn their place.
- **Discrete counts**: `SibSp`, `Parch`. Usable as-is, and combinable into a family size feature.
- **Identifier**: `PassengerId`. No predictive meaning whatsoever. Feeding row IDs to a model is a classic beginner error.
- **Free text**: `Name`. Useless raw, but it hides a genuinely valuable signal we will extract shortly.
- **Target**: `Survived`. Binary, complete, and never to be imputed.


---
# Section 2: Impute

### 2.1 Mean, median, and mode

The three foundational methods. All three replace gaps with a single constant computed from the column, which is both their strength (simple, fast, interpretable) and their weakness (variance shrinks, and relationships between features are ignored entirely).

Dropping rows is the tempting alternative, but it wastes information: `Age` is missing for 177 of 891 passengers, so `dropna()` would throw away 20% of an already small dataset. Worse, the rows with missing data are usually *systematically different* from complete rows, which we are about to prove.


In [ ]:
work = df.copy()   # always work on a copy, never the original

age_mean = work["Age"].mean()
age_median = work["Age"].median()
embarked_mode = work["Embarked"].mode()[0]

print("Mean age:  ", round(age_mean, 2))
print("Median age:", round(age_median, 2))
print("Modal port:", embarked_mode,
      "(.mode() returns a Series, [0] takes the first and breaks ties)")

mean_filled = work["Age"].fillna(age_mean)
median_filled = work["Age"].fillna(age_median)


### The variance problem, made visible

Mean imputation preserves the column mean exactly, which is its genuine selling point. What it does not preserve is the spread. Every constant-fill method compresses the distribution: 177 passengers all receive the identical age, so the data looks more clustered than reality.


In [ ]:
print("Mean of Age")
print("  original (gaps excluded):", round(work["Age"].mean(), 2))
print("  after mean imputation:   ", round(mean_filled.mean(), 2),
      "<- unchanged, this is mean imputation's one real strength")
print("  after median imputation: ", round(median_filled.mean(), 2))
print()
print("Standard deviation of Age")
print("  original (gaps excluded):", round(work["Age"].std(), 2))
print("  after mean imputation:   ", round(mean_filled.std(), 2))
print("  after median imputation: ", round(median_filled.std(), 2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].hist(work["Age"].dropna(), bins=40, color="steelblue",
             edgecolor="white")
axes[0].set_title("Age: original distribution")
axes[0].set_xlabel("Age"); axes[0].set_ylabel("Passengers")

axes[1].hist(mean_filled, bins=40, color="firebrick", edgecolor="white")
axes[1].axvline(age_mean, color="black", linestyle="--")
axes[1].set_title("Age: after mean imputation (note the spike)")
axes[1].set_xlabel("Age")
plt.tight_layout(); plt.show()


A spike at the mean that did not exist before. That artificial tower is 177 passengers who have been told they are all exactly 29.7 years old. This before-and-after comparison is a good habit, because a visible spike is your signal that the strategy distorted the data and something smarter may be warranted.

**Mode imputation** is the counterpart for categorical columns. You cannot average a port of embarkation, but you can take the most common one. Its weakness is the mirror image: it over-represents the dominant category, and it is close to meaningless when no category clearly dominates.


In [ ]:
print(work["Embarked"].value_counts())
print()
print("Southampton dominates at",
      round(work["Embarked"].value_counts(normalize=True).iloc[0] * 100, 1),
      "% so mode imputation is safe here.")
print("If the top three categories were roughly equal, the mode would be")
print("essentially arbitrary and would call for a smarter approach.")


### SimpleImputer, and why it beats fillna

Everything above used pandas `fillna()`, which is perfect for exploration. For an actual ML pipeline, use scikit-learn's `SimpleImputer` instead. The difference is not stylistic:

`fillna(df['Age'].mean())` computes the mean from **whatever data you hand it**. Run that on the full dataset before splitting and the training fill value contains information from the test set. `SimpleImputer` follows the fit and transform pattern instead: it *learns* the fill value from training data during `fit`, and applies that same stored value to any data you `transform`, including test data and production data arriving one row at a time.


In [ ]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy="median")        # also 'mean'
cat_imputer = SimpleImputer(strategy="most_frequent") # the mode

demo = work[["Age"]].copy()
num_imputer.fit(demo)                                  # learns from this data
print("Value learned during fit:", num_imputer.statistics_.round(2))
print("Missing after transform:",
      pd.DataFrame(num_imputer.transform(demo)).isnull().sum().sum())


### Validate, always

Two checks after any imputation: assert that nothing is missing, and confirm the distribution still looks like the original.


In [ ]:
filled = work.copy()
filled["Age"] = median_filled
filled["Embarked"] = filled["Embarked"].fillna(embarked_mode)

assert filled[["Age", "Embarked"]].isnull().sum().sum() == 0
print("Sanity check passed: zero missing values in the imputed columns.")


### 2.2 Domain-driven imputation

Mean, median, and mode know nothing about what the data means. They would assign the same age to a first class matron and a steerage child. Domain-driven imputation uses what we actually know about the passengers to make a better estimate, and on Titanic there is a beautiful source of that knowledge hiding in plain sight: the `Name` column contains each passenger's title.


In [ ]:
work["Title"] = work["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)

rare = work["Title"].value_counts()[lambda s: s < 10].index
work["Title"] = work["Title"].replace(list(rare), "Rare")

print(work["Title"].value_counts())
print()
print("Median age by title:")
print(work.groupby("Title")["Age"].median())


Look at that spread. "Master" was the formal address for a boy, and its median age is a handful of years. "Mrs" skews well above "Miss." A global median of 28 would age every missing child by two decades and misjudge nearly everyone else. Domain knowledge is measurably better information.

### Grouped imputation

The one-liner that puts it to work: compute the median *within* each meaningful group and fill from that, rather than from one global value.


In [ ]:
work["Age_grouped"] = (work.groupby(["Title", "Pclass"])["Age"]
                       .transform(lambda s: s.fillna(s.median())))

# Falling back gracefully: a group too small to have its own median leaves
# gaps behind, so a broader group (then the global median) catches them.
remaining = work["Age_grouped"].isnull().sum()
print("Still missing after Title plus Pclass grouping:", remaining)
work["Age_grouped"] = work["Age_grouped"].fillna(work["Age"].median())

comparison = pd.DataFrame({
    "global_median": median_filled[work["Age"].isnull()].head(8).values,
    "grouped": work.loc[work["Age"].isnull(), "Age_grouped"].head(8).values,
    "title": work.loc[work["Age"].isnull(), "Title"].head(8).values})
comparison


Same eight passengers, two very different sets of estimates. The global median hands everyone 28.0; the grouped version distinguishes a Master from a Mr from a Mrs. Choose grouping variables that genuinely predict the missing value: title and class predict age, but ticket number would not predict anything at all.

Watch for **small groups**. If a group has two members and one is missing, imputing from a single observation is barely better than guessing, which is why the fallback above matters: try the narrow group, then a broader one, then the global value.

### Time-based imputation

Titanic is not time-series data, but the technique belongs in your toolkit for the sensor and log data you will meet elsewhere. Forward fill carries the last known value forward on the assumption that state persists; interpolation draws a line between the known points on either side of the gap.


In [ ]:
sensor = pd.Series([21.0, np.nan, np.nan, 24.0, np.nan, 26.5],
                   index=pd.date_range("2026-08-24 09:00", periods=6, freq="h"),
                   name="temp_C")
pd.DataFrame({"raw": sensor,
              "ffill": sensor.ffill(),
              "bfill": sensor.bfill(),
              "interpolated": sensor.interpolate(method="time")})


**Always check the gap first.** Forward filling a ten minute sensor gap is reasonable; forward filling a three day outage is fiction dressed as data. The working rule: if the gap exceeds the data's natural frequency by more than about five times, do not forward fill. Leave it missing or choose a different strategy.

### KNN and model-based imputation

These estimate a missing value from the *other features* of that same row, rather than from a single column statistic.

- **KNNImputer** finds the k most similar complete rows and averages their values. A missing age is informed by the passenger's class, fare, and family size all at once.
- **IterativeImputer** (the MICE approach, Multivariate Imputation by Chained Equations) treats each incomplete feature as a regression target predicted from all the others, cycling through several rounds until the estimates settle.

Both need **scaled features first**, because KNN measures distance and a fare in the hundreds would otherwise drown out an age in the tens. Both are slower than a median. The sweet spot is moderate missingness (roughly 5 to 25%), meaningfully correlated features, and accuracy that matters more than speed.


In [ ]:
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler

knn_cols = ["Age", "Fare", "Pclass", "SibSp", "Parch"]
subset = work[knn_cols].copy()

# Scale first: KNN uses distances, so unscaled Fare would dominate
scaled = StandardScaler().fit_transform(subset)

knn_out = KNNImputer(n_neighbors=5).fit_transform(scaled)
mice_out = IterativeImputer(random_state=RANDOM_STATE,
                            max_iter=10).fit_transform(scaled)

print("Gaps before:", subset.isnull().sum().sum(),
      "| after KNN:", np.isnan(knn_out).sum(),
      "| after MICE:", np.isnan(mice_out).sum())

# Compare the strategies on the same passengers, back on the original scale
knn_age = (knn_out[:, 0] * subset["Age"].std() + subset["Age"].mean())
idx = work.index[work["Age"].isnull()][:6]
pd.DataFrame({"title": work.loc[idx, "Title"].values,
              "global_median": [round(age_median, 1)] * 6,
              "grouped": work.loc[idx, "Age_grouped"].round(1).values,
              "knn": knn_age[[work.index.get_loc(i) for i in idx]].round(1)})


Note the sklearn version detail: `from sklearn.experimental import enable_iterative_imputer` is required on older releases and harmless on newer ones, so the import above stays for portability.

### Indicator variables

Sometimes the fact that a value is missing is itself a signal. On Titanic, a missing age correlates with third class and with passengers nobody kept good records for, and that is information a model can use. The technique costs one column and is worth making a default habit: **record the missingness before you fill it**.


In [ ]:
work["Age_was_missing"] = work["Age"].isnull().astype(int)

print("Survival rate by whether Age was recorded:")
print(work.groupby("Age_was_missing")["Survived"].mean().round(3))
print()
print("Class distribution by whether Age was recorded:")
print(pd.crosstab(work["Age_was_missing"], work["Pclass"], normalize="index")
      .round(3))


Passengers with a missing age survived at a visibly lower rate, and they are heavily concentrated in third class. The gap is not random: it is explained by class, which makes this Missing at Random, and it carries real signal. The model now gets both the estimated age and the knowledge that it was estimated.

When missingness is Missing Not at Random, so that the gap itself encodes something no other column explains, the indicator becomes essential and the imputed value matters much less.

### Comparing strategies empirically

The best imputation strategy is the one that produces the best model, not the one that sounds most sophisticated. Make the choice empirical: build several versions of the dataset and compare, using cross-validation rather than a single split.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

base = work[["Pclass", "SibSp", "Parch", "Fare"]].copy()
base["Sex"] = (work["Sex"] == "female").astype(int)
y = work["Survived"]

variants = {
    "mean": mean_filled,
    "median": median_filled,
    "grouped by title and class": work["Age_grouped"],
    "grouped plus missing indicator": work["Age_grouped"],
}

for name, ages in variants.items():
    X = base.copy()
    X["Age"] = ages.values
    if "indicator" in name:
        X["Age_was_missing"] = work["Age_was_missing"].values
    score = cross_val_score(
        LogisticRegression(max_iter=1000), X, y, cv=5, scoring="f1").mean()
    print(name.ljust(32), "F1:", round(score, 4))


The differences are modest on this dataset, and that is a useful lesson in itself: sometimes the simple method wins and the sophisticated one is not worth its complexity. Sometimes it is worth several points. You only find out by measuring, and you should record which strategy won, by how much, and why you chose it. In production, revisit that comparison periodically, because distributions shift.

### 2.3 When NOT to impute

Two subsections on how to fill gaps; now the equally important question of when to stop. Not every missing value should be imputed, and knowing that is what separates careful work from running a cookbook recipe.

**Drop the column when it is mostly empty.** `Cabin` is 77% missing. Anything you fill in is more fiction than fact, and a model leaning on it is building on sand. The usual threshold is above 50%.


In [ ]:
print("Cabin missing:", round(work["Cabin"].isnull().mean() * 100, 1), "%")

# Before dropping, check whether the missingness itself is informative.
work["Had_cabin_record"] = work["Cabin"].notna().astype(int)
print("\nSurvival rate by whether a cabin was recorded:")
print(work.groupby("Had_cabin_record")["Survived"].mean().round(3))


This is the nuanced version of "drop it." The column is too empty to impute, but its *presence or absence* is strongly predictive, because recorded cabins belong overwhelmingly to first class passengers who were near the boat deck. So we discard the unusable values and keep the signal as an indicator. That is not a compromise, it is the correct answer.

**The other cases where dropping beats imputing:**

- **The target variable.** Never impute what you are trying to predict. If `Survived` were missing for a passenger, that row goes, full stop. Imputing the target means training the model on answers you invented.
- **Critical identifiers.** A missing primary key or join column means the record is corrupt. Drop it and go find out why your pipeline is producing broken records.
- **Low-value features.** A weak predictor that is also full of gaps is not worth the complexity. Drop it and simplify the pipeline.
- **Tiny percentages.** `Embarked` is missing for two passengers out of 891, well under 1%. Dropping those rows is safe, simple, and honest.

Dropping is not a failure. It is an honest acknowledgment that you do not have the data.


In [ ]:
print("Rows before:", len(work))
print("Missing target values:", work["Survived"].isnull().sum(),
      "(would be dropped, never imputed)")
print("Embarked missing:", work["Embarked"].isnull().sum(),
      "=", round(work["Embarked"].isnull().mean() * 100, 2), "% -> safe to drop")
print("Rows after dropping those:", len(work.dropna(subset=["Embarked"])))


### When imputation actively causes harm

- **Manufacturing false relationships.** Fill two columns with their means and every imputed row lands at the exact center of both distributions. Those rows become artificially similar to each other, creating a phantom cluster that never existed and that your model may happily discover.
- **Hiding data quality problems.** Imputation makes broken data look clean. The pipeline stops complaining and the underlying bug goes undetected.
- **Overconfident models.** The model cannot tell an observed value from an estimated one, unless you tell it with an indicator column.
- **Temporal traps.** Forward filling across a large gap invents continuity that never happened.
- **High-stakes decisions.** Medical, legal, and security contexts. Flagging a connection as malicious partly because of a made-up feature value means taking action against someone based on fiction.

Let's watch the phantom cluster form.


In [ ]:
both = work[["Age", "Fare"]].copy()
both.loc[rng.choice(len(both), 150, replace=False), "Fare"] = np.nan
imputed_both = both.fillna(both.mean())
was_missing = both.isnull().any(axis=1)

plt.figure(figsize=(8, 5))
plt.scatter(imputed_both.loc[~was_missing, "Age"],
            imputed_both.loc[~was_missing, "Fare"],
            s=14, alpha=0.35, color="steelblue", label="Observed")
plt.scatter(imputed_both.loc[was_missing, "Age"],
            imputed_both.loc[was_missing, "Fare"],
            s=26, alpha=0.8, color="firebrick", label="Contains an imputed value")
plt.ylim(0, 300)
plt.title("Mean imputation creates a cross of artificial values")
plt.xlabel("Age"); plt.ylabel("Fare"); plt.legend()
plt.show()


The red points line up along two straight lines that intersect at the pair of means. No such structure exists in reality. A distance-based model would see those rows as near neighbors of each other purely because they were incomplete in the same way.

**The robustness test:** run your analysis under several imputation strategies. If your conclusions change materially depending on which one you picked, then they are being driven by your imputation method rather than by your data, and the honest response is to get more data rather than to pick the strategy you like best.

### Alternatives to imputing at all

- **Algorithms that handle NaN natively.** XGBoost and LightGBM learn a default split direction for missing values during training, trying both branches and keeping whichever reduces loss. No imputation needed, and the missingness signal is exploited directly.
- **Separate models.** Train one model for rows where a feature is present and another for rows where it is absent. It sidesteps imputation entirely at the cost of maintaining two models.
- **Feature selection.** A column that is 40% missing and has low importance should simply be excluded.
- **Fix the pipeline upstream.** Missing values often mean a collection or ETL bug. Repairing a broken data feed beats any imputation algorithm. Treat missingness as a diagnostic signal about your infrastructure, not just a nuisance to be patched.

The best imputation is the one you never have to do.

### Missing data in production

Imputation is not a one-time cleaning step, it is a *learned parameter* that has to travel with your model. In production, rows arrive one at a time and you cannot recompute a median from a single row. You need the value calculated during training, which is exactly what `SimpleImputer` stores in `statistics_` and what a saved Pipeline carries with it automatically.

Two production concerns worth building in from the start. First, the edge case: what happens when a feature that was never missing during training suddenly arrives empty because an upstream source went offline? The model will return a prediction and give no error signal, which is why production imputation failures are silent and dangerous. Second, monitoring: track missing value rates per feature in production and alert when they deviate from training rates. A sudden spike means something upstream broke.

### The decision framework

| Situation | Action |
|---|---|
| Above 50% missing | Consider dropping the column, keep an indicator if presence is informative |
| Under 5% missing and random | Drop the rows or use a simple imputation |
| 5% to 30% missing | Domain-driven or grouped imputation, plus an indicator variable |
| Missing Not at Random | The indicator is essential; the filled value matters much less |
| Target variable missing | Never impute, drop the row |

Quantify how much is missing and where, diagnose why, decide per column rather than globally, validate by comparing distributions and model performance, and document every decision. In six months those notes are what let you debug a performance regression.


---
# Section 3: Handle outliers

Outliers are points that differ substantially from the rest of the data. They may be legitimate extreme values or they may be collection errors, and telling those apart is a judgment call that no algorithm makes for you. Models that rely on means and distances, such as linear regression and KNN, are especially sensitive to them.

`Fare` is our test subject: heavily right skewed, with a handful of passengers who paid more than 25 times the median.


In [ ]:
work = work.dropna(subset=["Embarked"]).reset_index(drop=True)
work["Age"] = work["Age_grouped"]        # adopt the grouped imputation

print(work["Fare"].describe().round(2))
print("\nThe five highest fares paid:")
print(work.nlargest(5, "Fare")[["Fare", "Pclass", "Ticket", "Survived"]])


Those top fares are not typos. Several passengers share ticket "PC 17755" at 512 pounds, a party travelling together on one ticket. This is the first and most important question in outlier work: **is this an error or a real extreme value?** Here it is real, which rules out deletion and points toward capping.

### 3.1 Z-score

The most intuitive detection method and the right place to start. The Z-score expresses how many standard deviations a value sits from the mean:

**Z = (x - mean) / standard deviation**

A Z of 0 is exactly average, a Z of 2 is two standard deviations above the mean. The conventional cutoff is **|Z| > 3**, because under a normal distribution only about 0.3% of values fall beyond three standard deviations.


In [ ]:
from scipy import stats

z = np.abs(stats.zscore(work["Fare"]))
work["Fare_zscore"] = z

for t in [2, 3, 4]:
    print("Threshold |Z| >", t, "flags", int((z > t).sum()), "passengers",
          "(" + str(round((z > t).mean() * 100, 2)) + "%)")


The threshold is a dial, not a law:

- **Z > 2** casts a wider net. Use it when missing an outlier is costly, as in security or safety work.
- **Z > 3** is the general purpose default.
- **Z > 4** is conservative. Use it in automated removal pipelines where a false positive means destroying real data.

Start at 3, plot the flagged points, see whether they look genuinely extreme or merely unusual, then adjust. Domain knowledge decides; there is no universally correct threshold.

### The masking effect

Z-score has a structural flaw, and it is a good one to understand deeply. The mean and standard deviation are themselves computed from data that *includes the outliers*. Extreme values drag the mean toward themselves and inflate the standard deviation, which shrinks every Z-score including their own. The outliers you most want to catch are the ones best equipped to hide.


In [ ]:
fare = work["Fare"]
without_extremes = fare[fare < 300]

print("With the extreme fares included:")
print("  mean:", round(fare.mean(), 2), "| std:", round(fare.std(), 2))
print("  Z-score of the 512 fare:",
      round((512.3292 - fare.mean()) / fare.std(), 2))
print()
print("With them excluded (what the bulk of the data actually looks like):")
print("  mean:", round(without_extremes.mean(), 2),
      "| std:", round(without_extremes.std(), 2))
print("  Z-score the 512 fare WOULD have had:",
      round((512.3292 - without_extremes.mean()) / without_extremes.std(), 2))


The most expensive ticket on the ship scores meaningfully lower against its own contaminated statistics than against clean ones. It is still flagged here, because it is extreme enough to survive the distortion, but it has understated its own strangeness by about two standard deviations. That is masking in one number, and a less extreme outlier, or a cluster of them, can understate itself right past the threshold. Multiple outliers compound it: a cluster can mask each other so effectively that none of them crosses the threshold.

### The Modified Z-score

The fix is to build the same measure out of statistics that outliers cannot move. Replace the mean with the median and the standard deviation with the **MAD** (Median Absolute Deviation), both of which depend on rank rather than magnitude:

**Modified Z = 0.6745 x (x - median) / MAD**

The 0.6745 constant rescales it to be comparable with ordinary Z-scores, and the conventional cutoff is **3.5**. This is a strong default for automated pipelines where you cannot eyeball the data first.


In [ ]:
mad = stats.median_abs_deviation(work["Fare"])
mod_z = 0.6745 * (work["Fare"] - work["Fare"].median()) / mad
work["Fare_modified_z"] = mod_z.abs()

print("Standard Z (>3) flags:        ", int((z > 3).sum()), "passengers")
print("Modified Z (>3.5) flags:      ", int((mod_z.abs() > 3.5).sum()), "passengers")
print("\nModified Z of the 512 fare:",
      round(0.6745 * (512.3292 - work['Fare'].median()) / mad, 1))


The modified version flags far more points, because it is not being fooled by its own targets. When Z-score and robust methods disagree substantially, masking is the reason, and you should trust the robust method.

### Per-group Z-scores

A global Z-score answers "unusual compared to everyone." Often the better question is "unusual **for this group**." A 30 pound fare is unremarkable in first class and extraordinary in third.


In [ ]:
work["Fare_z_by_class"] = (work.groupby("Pclass")["Fare"]
                           .transform(lambda s: (s - s.mean()) / s.std()).abs())

both_flags = work[(work["Fare_z_by_class"] > 3) & (work["Fare_zscore"] < 3)]
print("Passengers extreme WITHIN their class but not globally:", len(both_flags))
print(both_flags[["Pclass", "Fare", "Fare_zscore", "Fare_z_by_class"]]
      .head(4).round(2))


These passengers would slip past a global threshold entirely. The same principle drives behavioral security analytics: compute baselines per user, per server, or per time window rather than globally, because a user going from 5 to 50 requests a day is a tenfold jump that no global threshold will notice. Sliding window baselines (the last 30 days) adapt to evolving behavior better than all-time ones.

For count data such as logins or connections, apply a log transformation before computing Z-scores. It pulls right-skewed distributions toward symmetry, which is exactly the assumption Z-score needs.


In [ ]:
log_fare = np.log1p(work["Fare"])
print("Skew of Fare:          ", round(work["Fare"].skew(), 2))
print("Skew of log(1 + Fare): ", round(log_fare.skew(), 2),
      "<- much closer to symmetric, so Z-scores mean more")


**Z-score limitations, summarized:** it assumes rough normality and fails on skewed data, it is vulnerable to masking, and it is strictly univariate, examining one feature at a time. When it does not fit your distribution, reach for IQR.

### 3.2 Interquartile Range (IQR)

IQR uses percentiles rather than mean and standard deviation, which makes it robust by construction. If you remember one outlier method from this course, make it this one.

- **Q1** is the 25th percentile, **Q3** the 75th
- **IQR = Q3 - Q1**, the range spanned by the middle half of the data
- **Lower fence** = Q1 - 1.5 x IQR, **upper fence** = Q3 + 1.5 x IQR

Anything beyond the fences is an outlier. The 1.5 multiplier comes from John Tukey, who also invented the box plot.


In [ ]:
Q1, Q3 = work["Fare"].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

print("Q1:", round(Q1, 2), "| Q3:", round(Q3, 2), "| IQR:", round(IQR, 2))
print("Fences:", round(lower, 2), "to", round(upper, 2))
print("Flagged as outliers:", int((~work["Fare"].between(lower, upper)).sum()))


### Why IQR is robust

The concept is the **breakdown point**: the fraction of your data that can be replaced with arbitrary values before a statistic becomes meaningless. The mean has a breakdown point of zero, since a single extreme value can drag it anywhere. Quartiles each have a breakdown point of 25%, so up to a quarter of your data can be garbage and the quartiles still describe the bulk of it accurately.


In [ ]:
test = work["Fare"].copy()
print("Original    mean:", round(test.mean(), 2), "| Q3:", round(test.quantile(.75), 2))

test.iloc[0] = 1_000_000_000       # one absurd value
print("With 1e9 added:")
print("  mean:", round(test.mean(), 2), "<- destroyed")
print("  Q3:  ", round(test.quantile(.75), 2), "<- barely moved")


That is why IQR is safe as a first pass even before you know whether outliers exist. There is no circular reasoning, because the outliers cannot sabotage the mechanism detecting them.

### Adjusting the multiplier

- **1.5** is Tukey's default and works for most data
- **2.0 to 2.5** is conservative, appropriate for naturally skewed data such as response times or financial transactions
- **3.0** identifies only what Tukey called "far outliers" as opposed to "near outliers" at 1.5
- **1.0** is aggressive, for safety critical work where missing an outlier is expensive

If you are flagging 10% or more of your data, either the multiplier is too low or the distribution needs investigating. For right-skewed data, asymmetric fences are underused and effective: keep 1.5 on the lower side and loosen the upper side.


In [ ]:
for mult in [1.0, 1.5, 2.5, 3.0]:
    lo, hi = Q1 - mult * IQR, Q3 + mult * IQR
    n_out = int((~work["Fare"].between(lo, hi)).sum())
    print("multiplier", mult, "->", str(n_out).rjust(3), "outliers",
          "(" + str(round(n_out / len(work) * 100, 1)) + "%)")

# Asymmetric fences, respecting the natural right skew of fares
lo_asym, hi_asym = Q1 - 1.5 * IQR, Q3 + 2.5 * IQR
print("\nAsymmetric (1.5 lower, 2.5 upper):",
      int((~work["Fare"].between(lo_asym, hi_asym)).sum()), "outliers")


### Box plots are IQR made visible

Tukey invented both, and the box plot is a direct visual encoding of the method: the box spans Q1 to Q3, the line inside is the median, the whiskers extend to 1.5 times the IQR, and the dots beyond them are exactly the points IQR flags. Before-and-after box plots are the simplest quality check in outlier preprocessing.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot([work["Fare"]], vert=False, tick_labels=["Fare"])
axes[0].set_title("Fare: whiskers at 1.5 x IQR, dots are the flagged points")
axes[0].set_xlabel("Fare (pounds)")

work.boxplot(column="Fare", by="Pclass", ax=axes[1])
axes[1].set_title("Fare by passenger class")
axes[1].set_xlabel("Pclass"); axes[1].set_ylabel("Fare")
plt.suptitle(""); plt.tight_layout(); plt.show()


The grouped plot on the right makes the case for per-group thresholds: each class has its own natural range, and a single global fence over-flags one group while under-flagging another.

### IQR for feature engineering

Rather than removing flagged points, mark them. This keeps every row while giving the model an explicit signal.


In [ ]:
work["Fare_is_outlier"] = (~work["Fare"].between(lower, upper)).astype(int)

print("Survival rate by outlier flag:")
print(work.groupby("Fare_is_outlier")["Survived"].mean().round(3))

# Per-group fences: what counts as extreme depends on the class
def group_fences(s, mult=1.5):
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    return ~s.between(q1 - mult * iqr, q3 + mult * iqr)

work["Fare_outlier_in_class"] = (work.groupby("Pclass")["Fare"]
                                 .transform(group_fences).astype(int))
print("\nGlobal fences flag:  ", work["Fare_is_outlier"].sum())
print("Per-class fences flag:", work["Fare_outlier_in_class"].sum())


The flag is predictive on its own, which is a good reminder that outliers are not automatically noise to be scrubbed.

### 3.3 Winsorization

Removing outliers throws away rows. **Winsorization** tames them instead by capping at a boundary: values below the lower bound are set to the lower bound, values above the upper bound are set to the upper bound. Every data point survives; the extremes simply stop being extreme. The name comes from the biostatistician Charles Winsor, who argued that extreme values should be brought into line rather than discarded.

Common bounds are the 1st and 99th percentiles for gentle capping, or the 5th and 95th for aggressive reshaping. In pandas it is one call to `clip()`.


In [ ]:
p01, p99 = work["Fare"].quantile([0.01, 0.99])
p05, p95 = work["Fare"].quantile([0.05, 0.95])

work["Fare_wins_1_99"] = work["Fare"].clip(lower=p01, upper=p99)
work["Fare_wins_5_95"] = work["Fare"].clip(lower=p05, upper=p95)
work["Fare_wins_iqr"] = work["Fare"].clip(lower=lower, upper=upper)

print("Bounds: 1st/99th =", round(p01, 2), "/", round(p99, 2),
      "| 5th/95th =", round(p05, 2), "/", round(p95, 2))
print()
print(pd.DataFrame({
    "max": [work["Fare"].max(), work["Fare_wins_1_99"].max(),
            work["Fare_wins_5_95"].max(), work["Fare_wins_iqr"].max()],
    "mean": [work["Fare"].mean(), work["Fare_wins_1_99"].mean(),
             work["Fare_wins_5_95"].mean(), work["Fare_wins_iqr"].mean()],
    "rows": [len(work)] * 4},
    index=["original", "1st/99th", "5th/95th", "IQR fences"]).round(2))


Row count never changes. That is the whole point.

Note the **IQR-based bounds** option: using the Tukey fences as the capping boundaries combines the robustness of IQR detection with the data preservation of Winsorization, and it chooses the bounds from the data rather than from an arbitrary percentile.

Also worth tracking: **how many values got capped per feature**. A feature where 20% of values are being capped is telling you about a deeper data quality problem.


In [ ]:
for name, capped in [("1st/99th", work["Fare_wins_1_99"]),
                     ("5th/95th", work["Fare_wins_5_95"]),
                     ("IQR fences", work["Fare_wins_iqr"])]:
    n = int((capped != work["Fare"]).sum())
    print(name.ljust(11), "capped", str(n).rjust(3), "values",
          "(" + str(round(n / len(work) * 100, 1)) + "%)")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for ax, (title, series) in zip(axes, [
        ("Original", work["Fare"]),
        ("Winsorized at 1st/99th", work["Fare_wins_1_99"]),
        ("Winsorized at IQR fences", work["Fare_wins_iqr"])]):
    ax.hist(series, bins=45, color="steelblue", edgecolor="white")
    ax.set_title(title); ax.set_xlabel("Fare")
axes[0].set_ylabel("Passengers")
plt.tight_layout(); plt.show()


Always visualize before and after. If the shape has changed dramatically, reconsider the bounds.

### Winsorization versus trimming

| | Winsorization (capping) | Trimming (truncation) |
|---|---|---|
| Dataset size | Unchanged, every row preserved | Shrinks, rows are deleted |
| Extreme values | Become boundary values and still contribute | Contribute nothing |
| Assumption | None about the removed points | Removed points must be random or you introduce bias |
| Best for | Small datasets where every row matters | Clear errors you want gone entirely |

Conceptually, Winsorization says "these points exist but are not as extreme as they appeared," while trimming says "these points do not exist." On a small dataset the difference is material: trimming the top and bottom 5% of 891 rows costs roughly 89 observations.

### The leakage rule for Winsorization

This is the part people get wrong. **Percentile bounds must be computed on training data only.** Computing them across the full dataset lets test data influence the capping boundaries, which is the same leakage that fitting a scaler on everything would cause. Split first, then Winsorize.

The clean way to enforce that is a custom transformer whose `fit` learns the bounds and whose `transform` applies them. Because it follows scikit-learn's interface, it drops straight into a Pipeline alongside imputers and scalers.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class Winsorizer(BaseEstimator, TransformerMixin):
    """Cap features at percentile bounds learned from training data only."""

    def __init__(self, lower_pct=0.01, upper_pct=0.99):
        self.lower_pct = lower_pct
        self.upper_pct = upper_pct

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.lower_ = X.quantile(self.lower_pct)     # learned on TRAIN only
        self.upper_ = X.quantile(self.upper_pct)
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        return pd.DataFrame(X).clip(self.lower_, self.upper_, axis=1).values

    def get_feature_names_out(self, input_features=None):
        # Required so ColumnTransformer can report column names through
        # this step; a custom transformer that changes nothing about the
        # columns simply passes the incoming names along.
        if input_features is not None:
            return np.asarray(input_features, dtype=object)
        return self.feature_names_in_

wz = Winsorizer().fit(work[["Fare"]])
print("Bounds stored during fit:", round(wz.lower_.iloc[0], 2),
      "to", round(wz.upper_.iloc[0], 2))
print("Max after transform:", round(wz.transform(work[["Fare"]]).max(), 2))
print("These stored bounds are what gets applied to test and production data.")


Three patterns worth knowing:

1. **Winsorize then scale.** Cap the extremes before StandardScaler so they cannot inflate the standard deviation and compress everything else into a narrow band.
2. **Log-transform then Winsorize.** Reduce the skew first, then cap whatever remains. More effective for heavily right skewed features.
3. **Group-wise Winsorization.** Compute bounds per category. What is normal for a first class fare is extreme for third class.

### When to Winsorize, remove, or do neither

- **Winsorize** when extreme values are legitimate but disproportionate. The 512 pound fares are real bookings, so capping is right.
- **Remove** when values are impossible or clearly erroneous. Negative ages, future timestamps, physically impossible measurements.
- **Neither** when the outliers are the signal you are looking for. In anomaly detection, fraud detection, and intrusion detection, the outlier *is* the point. Scrubbing it destroys the thing you built the model to find.

That last one is the most important judgment call in outlier handling. Always ask first whether you are trying to reduce noise or detect signal.

### 3.4 Isolation Forest

Everything so far examines one feature at a time. Isolation Forest examines them together.

The key insight is elegant: **outliers are easier to isolate than normal points.** The algorithm builds random trees that repeatedly split the data on a random feature at a random value. A normal point sits in a crowd and takes many splits to separate from its neighbors. An outlier sits alone and falls out after one or two. Short average path length means anomaly, long path length means normal. It makes no assumptions about distribution shape and works on data of any form.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

iso_features = ["Age", "Fare", "Pclass", "SibSp", "Parch"]

# Scale FIRST: unscaled features with large ranges dominate the random splits
X_iso = StandardScaler().fit_transform(work[iso_features])

iso = IsolationForest(contamination=0.05, n_estimators=100,
                      max_samples="auto", random_state=RANDOM_STATE)
work["iso_flag"] = iso.fit_predict(X_iso)          # 1 = normal, -1 = outlier
work["iso_score"] = iso.decision_function(X_iso)   # lower = more anomalous

print(work["iso_flag"].value_counts().rename({1: "normal", -1: "outlier"}))
print("\nThe six most anomalous passengers:")
print(work.nsmallest(6, "iso_score")[
    ["Age", "Fare", "Pclass", "SibSp", "Parch", "iso_score"]].round(2))


The parameters worth knowing:

- **contamination**: your prior expectation of what fraction is anomalous, and usually the most impactful setting. It drives the decision threshold. Validate it against labeled examples when you have any, even a handful.
- **n_estimators**: number of trees, default 100. Raise it to 200 or 300 if scores fluctuate between runs.
- **max_samples**: samples drawn per tree, `'auto'` meaning the smaller of 256 and the dataset size. The small default is deliberate, since smaller subsamples make outliers easier to isolate.
- **max_features**: how many features each tree draws, controlling split diversity.
- **random_state**: always set it. Anomaly scores vary between runs otherwise.

### Interpreting anomaly scores

More negative means more anomalous. Scores near zero are borderline and could fall either way depending on threshold. Plotting the distribution often reveals a natural gap that makes a better threshold than whatever `contamination` assumed.


In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(work.loc[work["iso_flag"] == 1, "iso_score"], bins=45,
         color="steelblue", edgecolor="white", label="Normal")
plt.hist(work.loc[work["iso_flag"] == -1, "iso_score"], bins=25,
         color="firebrick", edgecolor="white", label="Flagged")
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.title("Anomaly score distribution: normal clusters tight, outliers tail left")
plt.xlabel("Anomaly score (lower is more anomalous)")
plt.ylabel("Passengers"); plt.legend()
plt.show()


Ranking by score rather than thresholding is how this gets used in security operations: the most suspicious items go to the top of the investigation queue, which is the only workable approach at high alert volumes. The scores themselves can also serve as a feature in a downstream supervised model, since they capture how unusual each row is relative to everything else.

**Pitfalls and practices:** scale features before fitting, remove known data entry errors first (you are hunting behavioral anomalies, not data quality problems), start with low contamination and raise it gradually, check score stability across random seeds, treat the output as a starting point for investigation rather than ground truth, and retrain periodically because last year's normal is not this year's.

### 3.5 Why multivariate outliers matter

Here is the case for all of this. A point can be perfectly normal in *every individual feature* and still be anomalous in combination.


In [ ]:
# Passengers that univariate methods miss but Isolation Forest catches
univariate_flagged = ((work["Fare_zscore"] > 3) |
                      (work["Fare_is_outlier"] == 1))
iso_flagged = work["iso_flag"] == -1

hidden = work[iso_flagged & ~univariate_flagged]
print("Flagged by Isolation Forest but NOT by any univariate check:",
      len(hidden))
print()
print(hidden[["Age", "Fare", "Pclass", "SibSp", "Parch"]].head(5).round(1))
print()
print("For reference, the typical passenger:")
print(work[["Age", "Fare", "SibSp", "Parch"]].median().round(1).to_dict())


In [ ]:
plt.figure(figsize=(9, 5.5))
normal = work[work["iso_flag"] == 1]
plt.scatter(normal["Age"], normal["Fare"], s=16, alpha=0.3,
            color="steelblue", label="Normal")
plt.scatter(hidden["Age"], hidden["Fare"], s=55, alpha=0.95,
            color="darkorange", edgecolor="black", linewidth=0.5,
            label="Multivariate outlier missed by univariate checks")
plt.axhline(upper, color="firebrick", linestyle="--", linewidth=1)
plt.text(1, upper + 12, "IQR upper fence for Fare", color="firebrick", fontsize=9)
plt.ylim(-10, 300)
plt.title("Normal on every axis alone, unusual in combination")
plt.xlabel("Age"); plt.ylabel("Fare")
plt.legend(loc="upper right")
plt.show()


The orange points sit comfortably below the fare fence and within a normal age range. No univariate method would look at them twice. What makes them unusual is the *combination*: an age, a fare, a class, and a family size that rarely occur together. Look at the rows above and the pattern is clear, large families travelling in third class on a shared ticket. Each attribute is ordinary on its own; five siblings and parents at that fare and that class is not.

This generalizes far beyond Titanic, and it is where the most dangerous patterns in data live:

- **Credit card fraud**: normal amount, normal time, but a new merchant category combined with a distant location
- **Network intrusion**: normal packet count, normal port, but an unusual protocol combined with a very short duration
- **Manufacturing**: normal temperature, normal pressure, but an abnormal vibration frequency alongside them signalling bearing failure
- **User behavior analytics**: a 3 AM login is normal for a night shift worker, but a 3 AM login from a foreign IP while accessing files never touched before is highly anomalous even though each individual feature is within bounds

Sophisticated attackers exploit exactly this, keeping every individual metric inside normal ranges while the overall pattern is plainly abnormal. The more features you have, the more combinations exist and the more it matters.

**The cost of ignoring it:** false negatives in security, distorted decision boundaries from multivariate outliers left in training data, misleading evaluation metrics, missed business insights (unusual customer segments can be untapped opportunities), and regulatory exposure in finance and healthcare where anomaly detection is increasingly required.

### Choosing a method

| Method | Use it for |
|---|---|
| Z-score | Quick screening on roughly normal distributions |
| Modified Z-score | The same, when masking is a concern or the pipeline is automated |
| IQR | The robust univariate default, any distribution |
| Winsorization | Taming extremes without discarding rows |
| Isolation Forest | Multivariate anomalies that appear only in combination |

In practice: IQR to clean obvious data quality problems, then Isolation Forest for subtle multivariate anomalies. And the governing principle throughout, **always investigate outliers before removing them**, because they may be your most valuable data points.

**Document all of it:** which method per feature and why, how many points were affected and what was done to them, the exact thresholds and parameters used (Z cutoff, IQR multiplier, contamination rate), all versioned alongside your model code. Six months from now, when performance degrades, that record is the difference between debugging and guessing. Revisit thresholds periodically, because distributions drift and, in security, adversaries actively adapt to whatever threshold you set.


---
# Section 4: Encode categorical data

ML algorithms need numbers, not text. As covered earlier, the *kind* of categorical determines the right conversion, and getting it wrong teaches the model relationships that do not exist.

### 4.1 Label and ordinal encoding

The simplest approach: assign each category a unique integer. It is fast and memory efficient, producing one column regardless of how many categories exist. It is also the most frequently misused technique in preprocessing, because those integers imply a mathematical ordering.


In [ ]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

le = LabelEncoder()
work["Embarked_label"] = le.fit_transform(work["Embarked"])

print("Mapping chosen by LabelEncoder:")
for i, c in enumerate(le.classes_):
    print("   ", c, "->", i)
print("\nNote: alphabetical, not meaningful. This tells a linear model that")
print("Southampton (2) is 'twice' Cherbourg (1), which is nonsense.")


That is the core hazard. Ports of embarkation have no order, so encoding them as 0, 1, 2 hands a linear or distance-based model a fictional relationship. A KNN model would compute distances between port numbers as though they meant something.

**Where label encoding is genuinely correct:**

- **Ordinal data**, where the ranking is real. `Pclass` is our example: first outranks second outranks third, and the integers already encode it.
- **Tree-based models**, which split on thresholds rather than interpreting magnitudes. Random Forest, Gradient Boosting, and XGBoost handle label encoded categoricals fine.
- **Target variables** in classification, where scikit-learn wants integer labels and the specific numbers do not affect behavior.

**Where it is dangerous:** linear regression, logistic regression, SVM, and KNN applied to nominal features. A useful diagnostic symptom: if model performance drops sharply when you switch from a tree-based model to a linear one, label encoded nominal features are a prime suspect.

### The alphabetical trap, and how to fix it

Suppose we had a severity-style ordinal column. The default alphabetical sort will scramble the order you intended.


In [ ]:
levels = ["Low", "Medium", "High", "Critical"]
demo_col = pd.DataFrame({"severity": ["High", "Low", "Critical", "Medium"]})

wrong = LabelEncoder().fit(demo_col["severity"])
print("Alphabetical (WRONG):",
      {c: i for i, c in enumerate(wrong.classes_)})
print("   This claims Low (2) is more severe than High (1).")

right = OrdinalEncoder(categories=[levels])
right.fit(demo_col[["severity"]])
print("\nExplicit order (CORRECT):",
      {c: i for i, c in enumerate(right.categories_[0])})


`OrdinalEncoder` also handles multiple columns in one step, which `LabelEncoder` cannot (it takes one column at a time and is really designed for target variables).

Two practical habits: use `inverse_transform` to recover original labels when debugging or interpreting predictions, and save the encoder (or at minimum `classes_`) so production systems can decode without re-fitting. And the rule that applies to every encoder: **fit on training data, then transform test data**, never fit on the combined dataset.


In [ ]:
encoded_sample = work["Embarked_label"].head(5).values
print("Encoded:", encoded_sample)
print("Decoded:", le.inverse_transform(encoded_sample))
print("\nStored classes to persist for production:", list(le.classes_))


A fourth mistake worth planning for now rather than at 2 AM: **unseen categories at inference time**. If a new port appears in production that was not in training, a naive encoder crashes. Either map unknowns to a reserved integer or use an encoder that handles them natively, which brings us to one-hot.

### 4.2 One-hot encoding

The universally safe choice: create a binary column per category. No ordering is implied, so every algorithm interprets it correctly. The cost is more columns.


In [ ]:
# pandas: quick and perfect for exploration
dummies = pd.get_dummies(work[["Embarked"]], columns=["Embarked"], dtype=int)
print(dummies.head(4))
print("\nNote dtype=int. pandas 2.0 returns bool by default, and int is")
print("what scikit-learn expects.")


`get_dummies` has a fatal flaw for production: it only knows the categories present in whatever data you hand it. If next month's data lacks one port, you get fewer columns than the model expects and the pipeline crashes. `OneHotEncoder` remembers the full training category set and, with `handle_unknown='ignore'`, turns an unseen category into an all-zero row instead of an exception.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ohe.fit(work[["Embarked", "Sex"]])

print("Feature names produced:", list(ohe.get_feature_names_out()))

# A category the encoder has never seen
unseen = pd.DataFrame({"Embarked": ["Z"], "Sex": ["female"]})
print("\nUnseen port 'Z' encodes to:", ohe.transform(unseen)[0].astype(int))
print("All zeros for the Embarked block, no crash. This is why production")
print("uses OneHotEncoder rather than get_dummies.")


Note `sparse_output=False` is the current parameter name; older scikit-learn versions called it `sparse`. Keep it False while debugging so you can read the values, and switch to sparse output for large datasets where memory matters.

### The drop_first parameter and multicollinearity

If a passenger did not board at Cherbourg and did not board at Queenstown, they must have boarded at Southampton. The third column is perfectly predictable from the other two, which is **multicollinearity**. It does not break predictions, but for plain linear and logistic regression it produces unstable coefficients and inflated standard errors, so individual coefficients cannot be trusted for interpretation.

Dropping one category turns it into a reference: the remaining coefficients then read as "difference from Southampton."


In [ ]:
full = pd.get_dummies(work["Embarked"], prefix="Emb", dtype=int)
reduced = pd.get_dummies(work["Embarked"], prefix="Emb", drop_first=True,
                         dtype=int)
print("Without drop_first:", list(full.columns))
print("With drop_first:   ", list(reduced.columns),
      "(the dropped one is the reference)")


When to use it: yes for plain linear and logistic regression, unnecessary for tree-based models where multicollinearity is a non-issue, and optional for regularized models like Lasso and Ridge which handle it naturally. When in doubt include it, because dropping one column never hurts predictive performance.

### Cardinality decides everything

One-hot is the clear default below roughly 10 to 15 categories. Between 15 and 100, weigh whether the column explosion is worth it. Above 100, it stops being viable.


In [ ]:
for col in ["Sex", "Embarked", "Cabin", "Ticket"]:
    n = work[col].nunique()
    verdict = "one-hot" if n <= 15 else "target or frequency encoding"
    print(col.ljust(9), str(n).rjust(3), "categories ->", verdict)


`Ticket` has 680 distinct values. One-hot encoding it would add 680 mostly-empty columns to a dataset of 889 rows, which is exactly the trap. The security equivalent is one-hot encoding raw port numbers: 65,535 columns is absurd, and the right move is to bin them into meaningful groups (well-known, registered, dynamic) and one-hot the bins.

Let's apply that binning idea to `Cabin`, where the deck letter is the meaningful part.


In [ ]:
# Take the deck letter, and make the absence explicit rather than null.
# Note: do NOT reach for .astype(str) here. In current pandas it leaves
# NaN as NaN, so the nulls would survive and get quietly imputed later.
work["Deck"] = work["Cabin"].str[0].fillna("Unknown")

print(work["Deck"].value_counts())
print()
print(work["Cabin"].nunique(), "cabin values collapse to",
      work["Deck"].nunique(), "decks, which one-hot handles comfortably.")
assert work["Deck"].isnull().sum() == 0, "Deck still has nulls!"


### 4.3 Target encoding, and its leakage risk

For genuinely high-cardinality columns, target encoding is the power tool: replace each category with the **mean of the target for that category**. A thousand categories compress into one column that directly expresses each category's relationship to the outcome.

And it is dangerous, because you are building a feature out of the answer.


In [ ]:
# The naive version, deliberately done wrong so we can see the failure
naive = work.groupby("Ticket")["Survived"].mean()
work["Ticket_target_naive"] = work["Ticket"].map(naive)

ticket_counts = work["Ticket"].value_counts()
singletons = ticket_counts[ticket_counts == 1].index

print("Tickets held by exactly one passenger:", len(singletons),
      "out of", work["Ticket"].nunique())
print()
print("What the naive encoding gives those passengers:")
print(work[work["Ticket"].isin(singletons)]["Ticket_target_naive"]
      .value_counts())


Every single-passenger ticket gets encoded as exactly 0.0 or 1.0, which is that passenger's own survival value copied into a feature. The model is not learning a pattern, it is reading the answer off the back of the card. Train on this and you will see spectacular validation scores followed by production failure.

### Smoothing

The fix is to blend each category's mean with the global mean, weighted by how much evidence the category actually has:

**encoded = (n x category_mean + m x global_mean) / (n + m)**

where `n` is the number of samples in the category and `m` is the smoothing parameter. Large categories keep their own mean because `n` dominates. Small categories get pulled toward the global mean, so they cannot overfit to a handful of observations.


In [ ]:
global_mean = work["Survived"].mean()

def smoothed_target_encode(series, target, m=20):
    stats_ = target.groupby(series).agg(["mean", "count"])
    smoothed = ((stats_["count"] * stats_["mean"] + m * global_mean)
                / (stats_["count"] + m))
    return series.map(smoothed)

work["Ticket_target_smooth"] = smoothed_target_encode(
    work["Ticket"], work["Survived"], m=20)

print("Global survival rate:", round(global_mean, 3))
print()
print("Singleton tickets, naive vs smoothed:")
print(pd.DataFrame({
    "naive": work.loc[work["Ticket"].isin(singletons),
                      "Ticket_target_naive"].head(6).values,
    "smoothed": work.loc[work["Ticket"].isin(singletons),
                         "Ticket_target_smooth"].head(6).round(3).values}))


The smoothed values sit close to the global rate, which is the honest statement: one passenger tells us almost nothing about that ticket. If you implement this by hand, start with `m` somewhere between 10 and 30 and adjust if rare categories still produce extreme values.

scikit-learn's built-in `TargetEncoder` (available from version 1.3) handles smoothing automatically using internal cross-validation, so the encoding for each row is computed from folds that exclude it. Inside a Pipeline, it fits on training data only.


In [ ]:
from sklearn.preprocessing import TargetEncoder

te = TargetEncoder(smooth="auto", random_state=RANDOM_STATE)
te_out = te.fit_transform(work[["Deck"]], work["Survived"])

print("Deck encoded by survival rate:")
for deck, val in zip(te.categories_[0], te.encodings_[0]):
    print("   ", str(deck).ljust(8), round(float(val), 3))


Look at deck T, which has exactly one passenger. Automatic smoothing shrinks rare categories toward the global rate but cannot rescue a category of one, so treat any encoded value built on a handful of rows with suspicion regardless of who computed it. Note also that the deck encodings all sit well above the global survival rate of 0.38, because a recorded cabin overwhelmingly means first class.

### Frequency encoding, the leakage-safe alternative

Replace each category with how often it appears. The target is never touched, so there is zero leakage risk, and it works surprisingly well when rarity itself is predictive. That is common in security: a user agent string appearing twice across millions of requests is inherently suspicious.


In [ ]:
freq = work["Ticket"].value_counts(normalize=True)
work["Ticket_frequency"] = work["Ticket"].map(freq)

print("Ticket frequency encoding, sample:")
print(work[["Ticket", "Ticket_frequency"]].head(4).round(4))
print()
print("Survival rate for shared tickets vs solo tickets:")
work["shared_ticket"] = (work["Ticket"].map(work["Ticket"].value_counts()) > 1)
print(work.groupby("shared_ticket")["Survived"].mean().round(3))


Rarity carries real signal here: passengers on shared tickets (families and groups) survived at a noticeably different rate than solo travellers, and frequency encoding captured that without ever looking at the target.

**Binary encoding** is the third option for high cardinality: label encode, then represent those integers in binary digits across several columns. One hundred categories need only seven columns instead of one hundred. The tradeoff is that it introduces some artificial relationships between the digit columns. Combining frequency and target encoding on the same column often beats either alone.

### The encoding decision framework

| Feature type | Encoding |
|---|---|
| Binary | Direct 0/1 mapping, no library needed |
| Ordinal, under 10 levels | Ordinal encoding with explicit category order |
| Nominal, under 15 categories | One-hot, always safe for any algorithm |
| High cardinality, 15+ | Target encoding with smoothing, or frequency encoding |
| Any, with tree-based models only | Label encoding is fine and keeps the feature count low |
| Any, with linear models | One-hot for low cardinality, target encoding for high |

Always validate empirically rather than assuming the most sophisticated method wins, and document every encoding choice, because those decisions are part of your model's logic and determine what patterns it can learn.


---
# Section 5: Scale

Our features live on wildly different scales: `Age` runs 0 to 80, `Fare` runs 0 to 512, `Pclass` is 1 to 3, and one-hot columns are 0 or 1. Many algorithms treat the large-magnitude feature as the important one purely because of its units.

**Algorithms that need scaling:** KNN, SVM, linear and logistic regression, neural networks, and PCA.
**Algorithms that do not:** decision trees, random forest, gradient boosting, and XGBoost, because they split on thresholds and the threshold simply adjusts to whatever scale it finds.

Scaling does not change the information in your data, only its representation.

### 5.1 StandardScaler

**x_scaled = (x - mean) / standard deviation**

Every feature ends up with mean 0 and standard deviation 1. This is the Z-score transformation applied as a preprocessing step, and it is the most commonly used scaler in practice. After scaling, a value of 2 means "two standard deviations above average" no matter which feature it came from.


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

num_cols = ["Age", "Fare", "SibSp", "Parch"]
X_num = work[num_cols]

std_scaled = pd.DataFrame(StandardScaler().fit_transform(X_num),
                          columns=num_cols)

print("Before scaling:")
print(X_num.describe().loc[["mean", "std", "min", "max"]].round(2))
print("\nAfter StandardScaler (mean 0, std 1):")
print(std_scaled.describe().loc[["mean", "std", "min", "max"]].round(2))


Notice that `Fare` still reaches beyond 9 after scaling. **StandardScaler preserves outliers**: a value that was ten standard deviations out before is still ten standard deviations out after. Scaling is not outlier handling, which is why the capping work in the previous section comes first.

### 5.2 MinMaxScaler

**x_scaled = (x - min) / (max - min)**

Everything is squeezed into the range 0 to 1. Use it when you need bounded values, which mainly means neural networks with sigmoid or tanh activations, or image data. `MinMaxScaler(feature_range=(-1, 1))` gives symmetric bounds.

Its critical weakness is outlier sensitivity, and `Fare` demonstrates it perfectly.


In [ ]:
mm_scaled = pd.DataFrame(MinMaxScaler().fit_transform(X_num),
                         columns=num_cols)

print("Where MinMaxScaler puts the Fare distribution:")
print(mm_scaled["Fare"].describe().round(4))
print()
print("The median fare lands at",
      round(mm_scaled["Fare"].median(), 4),
      "because one 512 pound ticket defines the top of the range.")
print("Nearly every passenger is compressed into the bottom few percent.")


One extreme value has flattened the entire distribution. StandardScaler is less vulnerable because it uses the mean and standard deviation rather than the min and max, and this is precisely why outlier handling belongs before scaling in the pipeline order.

### 5.3 RobustScaler

When outliers cannot be removed but scaling is still required, RobustScaler centers on the **median** and scales by the **IQR**, both of which extreme values barely move.


In [ ]:
rb_scaled = pd.DataFrame(RobustScaler().fit_transform(X_num),
                         columns=num_cols)

comparison = pd.DataFrame({
    "StandardScaler": std_scaled["Fare"].describe(),
    "MinMaxScaler": mm_scaled["Fare"].describe(),
    "RobustScaler": rb_scaled["Fare"].describe()}).round(3)
comparison.loc[["mean", "std", "50%", "max"]]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, (name, series) in zip(axes, [
        ("StandardScaler", std_scaled["Fare"]),
        ("MinMaxScaler", mm_scaled["Fare"]),
        ("RobustScaler", rb_scaled["Fare"])]):
    ax.hist(series, bins=45, color="steelblue", edgecolor="white")
    ax.set_title(name); ax.set_xlabel("Scaled Fare")
axes[0].set_ylabel("Passengers")
plt.tight_layout(); plt.show()


Same data, same information, three different representations. The shape never changes, only the axis.

### 5.4 The fit and transform pattern

This is the most important discipline in this section.

- **`fit()`** learns parameters from data: the mean and standard deviation, or the min and max
- **`transform()`** applies those stored parameters to any dataset
- **`fit_transform()`** does both, and belongs **only** on training data

Calling `fit()` or `fit_transform()` on test data is data leakage: the test set's statistics influence the transformation, so the model has indirectly seen data it will be evaluated on, and your metrics become optimistically biased.


In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te = train_test_split(X_num, test_size=0.25,
                              random_state=RANDOM_STATE)

scaler = StandardScaler()
tr_scaled = scaler.fit_transform(X_tr)     # LEARN from train, and transform it
te_scaled = scaler.transform(X_te)         # APPLY those same parameters

print("Parameters learned from training data:")
print("  means:", scaler.mean_.round(2))
print("  stds: ", np.sqrt(scaler.var_).round(2))
print()
print("Training data after scaling: mean", round(tr_scaled.mean(), 4),
      "(exactly 0, as designed)")
print("Test data after scaling:     mean", round(te_scaled.mean(), 4),
      "(near but not exactly 0, which is CORRECT)")
print("\nIf the test mean were exactly 0, the scaler had seen the test set.")


That last line is the tell. Correctly scaled test data does not center perfectly, because it was transformed with parameters it did not contribute to.

### 5.5 Edge cases

- **New data outside the training range.** MinMaxScaler will happily produce values above 1. If training saw a maximum of 100 and production sends 150, the scaled value is 1.5. Expected behavior, not a bug.
- **Zero-variance features.** StandardScaler divides by the standard deviation. scikit-learn detects a zero and sets the output to 0 rather than raising an error, but a feature with no variance carries no information and should probably be dropped anyway.
- **Sparse data.** StandardScaler subtracts the mean from every value, which destroys sparsity and can explode memory. Use `StandardScaler(with_mean=False)` or `MaxAbsScaler` instead.
- **Recovering original units.** Both scalers support `inverse_transform()`.
- **Drift.** Production distributions shift over time, so scalers need periodic retraining just like models.


In [ ]:
flat = pd.DataFrame({"constant": [7.0] * 100, "normal": rng.normal(0, 1, 100)})
out = StandardScaler().fit_transform(flat)
print("Zero-variance column after scaling:", np.unique(out[:, 0]),
      "<- set to 0, no divide-by-zero error")

original = scaler.inverse_transform(te_scaled)
print("\ninverse_transform recovers the original units:",
      np.allclose(original, X_te.values))


### 5.6 When scaling actually matters

Rather than take this on faith, measure it. KNN is distance-based, so it should be highly sensitive; a decision tree splits on thresholds, so it should be indifferent.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

X_demo = work[["Age", "Fare", "Pclass", "SibSp", "Parch"]]
y_demo = work["Survived"]

for name, model in [("KNN (distance-based)", KNeighborsClassifier()),
                    ("Decision tree (threshold-based)",
                     DecisionTreeClassifier(random_state=RANDOM_STATE))]:
    raw = cross_val_score(model, X_demo, y_demo, cv=5, scoring="accuracy").mean()
    scaled = cross_val_score(
        Pipeline([("scale", StandardScaler()), ("model", model)]),
        X_demo, y_demo, cv=5, scoring="accuracy").mean()
    print(name.ljust(34), "unscaled:", round(raw, 4),
          "| scaled:", round(scaled, 4),
          "| change:", ("+" if scaled >= raw else "") +
          str(round((scaled - raw) * 100, 2)) + " pts")


KNN improves substantially. The tree barely moves, exactly as theory predicts: without scaling, KNN's distance calculations are dominated by `Fare` because a 50 pound fare difference swamps a 50 year age difference numerically, even though age is the more useful signal.

### Convergence

Gradient-based algorithms have a second reason to scale. Unscaled features create an elongated loss surface, a narrow canyon rather than a bowl, so gradient descent zigzags across the narrow dimension instead of heading for the bottom. It also makes learning rate selection much harder, because no single rate suits features on very different scales.


In [ ]:
from sklearn.linear_model import LogisticRegression

raw_model = LogisticRegression(solver="saga", max_iter=5000,
                               random_state=RANDOM_STATE).fit(X_demo, y_demo)
scaled_model = LogisticRegression(solver="saga", max_iter=5000,
                                  random_state=RANDOM_STATE).fit(
    StandardScaler().fit_transform(X_demo), y_demo)

print("Iterations to converge, unscaled:", raw_model.n_iter_[0])
print("Iterations to converge, scaled:  ", scaled_model.n_iter_[0])


### Regularization makes scaling mandatory

L1 (Lasso) and L2 (Ridge) penalties are applied to coefficient *magnitude*. Without scaling, a fare coefficient might be 0.0001 while an age coefficient is 5.0, purely because of their units. Regularization would then penalize the age coefficient roughly fifty thousand times more heavily, not because age matters less but because its scale demands a bigger number. A genuinely predictive small-scale feature can be zeroed out by Lasso for no reason other than its units.

After scaling, coefficients reflect predictive contribution and regularization can do its actual job. For regularized models this is not a preference, it is a mathematical requirement. The same applies to ElasticNet.

### Mixed feature types

Real datasets need different treatment per column, which is exactly what `ColumnTransformer` is for:

- **Numeric features**: StandardScaler or MinMaxScaler
- **One-hot columns**: already 0/1, so scaling adds nothing. Some practitioners scale for consistency, others leave them
- **Ordinal features**: scale if the range is large or the intervals are uneven; leave a compact 1 to 5 range alone
- **Binary features**: already minimal, no benefit

**The decision checklist:** identify whether your algorithm is distance-based, gradient-based, or regularized (then scale) or tree-based or probability-based (optional); choose the scaler from the data (normal to StandardScaler, bounded requirement to MinMaxScaler, heavy outliers to RobustScaler); fit on training only and transform everything; then verify the scaled distributions. When in doubt, scale. It costs almost nothing and never hurts tree models.

### One caution about interpretability

Scaled coefficients lose their original units. "A one year increase in age" is meaningful to a stakeholder; "a one standard deviation increase" is less intuitive. When communicating results matters more than squeezing out accuracy, that tradeoff is worth weighing.


---
# Section 6: Assemble the pipeline

Everything so far was done step by step so you could see it work. That is not how you ship it. Done manually, you must remember to apply every transformation to training data, then apply the *identical* transformations to test data, then again in production, in the right order, without ever fitting on the wrong subset. Each of those is a chance to introduce a silent bug.

A `Pipeline` bundles preprocessing and model into one object that enforces correct ordering and makes preprocessing leakage structurally impossible. It is the professional standard, not a nice-to-have.

### The design

- **Numeric columns** (`Age`, `Fare`, `SibSp`, `Parch`): impute the median, cap the extremes, then scale
- **Categorical columns** (`Sex`, `Embarked`, `Deck`): impute the mode, then one-hot encode
- **Ordinal column** (`Pclass`): already correctly ordered integers, pass through
- **Indicator columns** we engineered: pass through
- **Everything else** (`PassengerId`, `Name`, `Ticket`, `Cabin`): dropped, which `ColumnTransformer` does by default


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score

model_df = work.copy()   # Age_was_missing and Had_cabin_record already engineered

numeric_features = ["Age", "Fare", "SibSp", "Parch"]
categorical_features = ["Sex", "Embarked", "Deck"]
passthrough_features = ["Pclass", "Age_was_missing", "Had_cabin_record"]

X = model_df[numeric_features + categorical_features + passthrough_features]
y = model_df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

print("Train:", X_train.shape, "| Test:", X_test.shape)


In [ ]:
# Impute -> Winsorize -> Scale
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("winsorize", Winsorizer(lower_pct=0.01, upper_pct=0.99)),
    ("scale", StandardScaler()),
])

# Impute -> One-hot encode
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
        ("pass", "passthrough", passthrough_features),
    ],
    remainder="drop",       # anything unlisted is dropped, the safe default
)

pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

pipe


One object now contains the entire workflow. `remainder='drop'` is the default and the safer choice for production, guaranteeing only explicitly processed features reach the model; `remainder='passthrough'` keeps unlisted columns unchanged, which is handy during development.

### Fit once, and everything happens in the right order


In [ ]:
pipe.fit(X_train, y_train)

print("Training accuracy:", round(pipe.score(X_train, y_train), 4))
print("Test accuracy:    ", round(pipe.score(X_test, y_test), 4))
print()
print("What one call to .fit() just did:")
print("  1. Learned the median of each numeric column from TRAINING data")
print("  2. Learned Winsorization bounds from TRAINING data")
print("  3. Learned scaling parameters from TRAINING data")
print("  4. Learned the category set and the mode from TRAINING data")
print("  5. Trained the classifier on the result")
print("And .score(X_test) applied all of those stored parameters unchanged.")


### Cross-validation without leakage

This is where pipelines earn their keep. Preprocess manually before cross-validating and every validation fold has already influenced the statistics used to transform it, so your scores come back optimistically biased. Hand the pipeline to `cross_val_score` and each fold re-fits preprocessing from scratch on that fold's training portion only.


In [ ]:
scores = cross_val_score(pipe, X, y, cv=5, scoring="f1")
print("F1 per fold:", scores.round(4))
print("Mean F1:", round(scores.mean(), 4), "+/-", round(scores.std(), 4))
print()
print("Each fold got its own imputation values, its own Winsorization bounds,")
print("its own scaling parameters, and its own category set. Zero leakage.")


### Tuning preprocessing and model together

`GridSearchCV` accepts the whole pipeline, and the double underscore convention addresses any parameter at any step: `stepname__parametername`. That means you can tune *preprocessing choices* as hyperparameters alongside model settings, which is the honest way to decide between a median and a mean.


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "preprocess__num__impute__strategy": ["mean", "median"],
    "preprocess__num__winsorize__lower_pct": [0.0, 0.01, 0.05],
    "model__C": [0.1, 1.0, 10.0],
}

search = GridSearchCV(pipe, param_grid, cv=5, scoring="f1", n_jobs=-1)
search.fit(X_train, y_train)

print("Best F1:", round(search.best_score_, 4))
print("Best settings:")
for k, v in search.best_params_.items():
    print("   ", k, "=", v)


The grid searched imputation strategy, capping aggressiveness, and regularization strength together, and every combination was evaluated with fresh preprocessing per fold.

### Saving and deploying

`joblib.dump` writes one file containing everything: imputer statistics, Winsorization bounds, encoder mappings, scaler parameters, and model weights. Production loads that single object, calls `predict` on raw data, and returns results. There is no separate preprocessing code to maintain or to drift out of sync with training, which eliminates an entire category of deployment bug.


In [ ]:
import joblib

best_pipe = search.best_estimator_
joblib.dump(best_pipe, "models/titanic_pipeline_2026-08-24_v1.pkl")

loaded = joblib.load("models/titanic_pipeline_2026-08-24_v1.pkl")
print("Reloaded pipeline test accuracy:",
      round(loaded.score(X_test, y_test), 4))
print("File contents: imputer stats, capping bounds, encoder mappings,")
print("scaler parameters, and model weights, all in one artifact.")


Two cautions. **Version your pipelines**, with the training date and version in the filename, stored alongside your code. And a security warning: **pickle and joblib files execute arbitrary code when loaded**. Never load a pipeline from an untrusted source, the same discipline you would apply to running any unknown executable.

### The production edge case

Remember the concern about a feature that was never missing during training suddenly arriving empty. The real Kaggle Titanic test set contains exactly one passenger with a missing fare, so let's see what our pipeline does with a row like that.


In [ ]:
new_passenger = pd.DataFrame([{
    "Age": 60.5, "Fare": np.nan,          # fare missing, as in the real test set
    "SibSp": 0, "Parch": 0,
    "Sex": "male", "Embarked": "S", "Deck": "Unknown",
    "Pclass": 3, "Age_was_missing": 0, "Had_cabin_record": 0}])

pred = loaded.predict(new_passenger)[0]
prob = loaded.predict_proba(new_passenger)[0][1]

print("Prediction:", "survived" if pred == 1 else "did not survive",
      "| probability:", round(prob, 3))
print()
print("The pipeline filled the missing fare with the median it learned during")
print("training, then capped, scaled, and encoded exactly as it did in training.")
print("No error, no special handling, no separate preprocessing code.")
print()
print("The flip side: it returned a confident answer for a row with an invented")
print("feature value and gave no warning. This is why production imputation")
print("failures are silent, and why you monitor missing rates per feature.")


---
# Section 7: Inspect the transformed data

A pipeline that works is not the same as a pipeline you understand. Column names vanish into a NumPy array by default, so verifying what actually came out is the final step.


In [ ]:
feature_names = best_pipe.named_steps["preprocess"].get_feature_names_out()

print("Input columns: ", X_train.shape[1])
print("Output columns:", len(feature_names))
print()
for name in feature_names:
    print("   ", name)


One-hot encoding expanded three categorical columns into a wider set of binary indicators, which accounts for the growth. The prefixes tell you which transformer produced each column.

### Getting a DataFrame back instead of an array

From scikit-learn 1.2 onward, `set_output(transform='pandas')` preserves column names all the way through, which makes inspection and debugging far easier.


In [ ]:
inspect_pipe = Pipeline([("preprocess", preprocessor)])
inspect_pipe.set_output(transform="pandas")
inspect_pipe.fit(X_train, y_train)

transformed = inspect_pipe.transform(X_train)
print("Type:", type(transformed).__name__, "| shape:", transformed.shape)
transformed.head(4).round(3)


### Sanity checks on the output

Four things to verify before any transformed dataset goes to a model.


In [ ]:
print("1. No missing values:      ",
      "PASS" if transformed.isnull().sum().sum() == 0
      else "FAIL, " + str(transformed.isnull().sum().sum()) + " remain")

num_out = [c for c in transformed.columns if c.startswith("num__")]
means = transformed[num_out].mean().abs().max()
stds = transformed[num_out].std()
print("2. Numerics centered:       ",
      "PASS" if means < 0.01 else "FAIL", "(max |mean| =", round(means, 5), ")")
print("3. Numerics unit variance:  ",
      "PASS" if np.allclose(stds, 1, atol=0.02) else "FAIL",
      "(std range", round(stds.min(), 3), "to", round(stds.max(), 3), ")")

cat_out = [c for c in transformed.columns if c.startswith("cat__")]
binary_ok = transformed[cat_out].isin([0.0, 1.0]).all().all()
print("4. One-hot columns binary:  ", "PASS" if binary_ok else "FAIL")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))

axes[0].hist(X_train["Fare"], bins=45, color="gray", edgecolor="white")
axes[0].set_title("Fare: raw input")
axes[0].set_xlabel("Pounds")

axes[1].hist(transformed["num__Fare"], bins=45, color="steelblue",
             edgecolor="white")
axes[1].set_title("Fare: imputed, capped, scaled")
axes[1].set_xlabel("Standard deviations")

axes[2].bar(range(len(cat_out)), transformed[cat_out].mean().values,
            color="steelblue")
axes[2].set_xticks(range(len(cat_out)))
axes[2].set_xticklabels([c.replace("cat__", "") for c in cat_out],
                        rotation=90, fontsize=7)
axes[2].set_title("One-hot column frequencies")
axes[2].set_ylabel("Proportion")

plt.tight_layout(); plt.show()


The middle panel is the whole section in one picture. The raw fare distribution had a long tail stretching to 512 and a scale nobody else shared. After imputation, capping, and scaling it is centered near zero, measured in standard deviations, and bounded by the capping we chose deliberately rather than by whatever the most expensive ticket happened to cost.

### What the model actually learned

Because the features are scaled, the coefficients are now directly comparable to each other, which is one of the underrated benefits of scaling.


In [ ]:
coefs = pd.Series(best_pipe.named_steps["model"].coef_[0],
                  index=feature_names).sort_values(key=abs, ascending=False)
print("Strongest influences on the prediction:")
print(coefs.head(8).round(3))


Sex dominates, then class and fare, which matches everything known about that night. Coefficients this readable are only possible because every feature is on the same scale.


In [ ]:
out_path = "data/processed/titanic_2026-08-24_transformed.csv"
transformed.to_csv(out_path, index=False)
print("Saved transformed training features to:", out_path)
print("Raw data remains untouched at data/raw/titanic.csv")


---
# Wrap-up

You took a raw passenger manifest with a 77% empty column, a fifth of its ages missing for reasons that were not random, a fare distribution stretching across three orders of magnitude, and categoricals ranging from 2 to 681 distinct values, and turned it into a clean numeric matrix that any algorithm can consume.

The order of operations matters and is worth memorizing: **split, then impute, then handle outliers, then encode, then scale, then model.** Splitting comes first so that every subsequent step learns only from training data. Outlier capping precedes scaling so extremes cannot distort the scaling parameters. Encoding precedes scaling because you cannot scale text.

The judgment calls mattered more than the code:

- Dropping `Cabin` while keeping its presence as an indicator, because a column too empty to impute can still be too informative to discard entirely
- Imputing `Age` by title and class rather than one global median, because a Master is not a Mr
- Capping fares rather than deleting those passengers, because the 512 pound tickets were real
- Reaching for target encoding on `Ticket` and then immediately defending against the leakage it invites
- Recognizing that on this dataset the sophisticated imputation barely beat the simple one, and that measuring is what told us so

Every one of those decisions is now baked into a single artifact that takes raw data in and produces predictions out, with no separate preprocessing code to drift out of sync.

**Next up, Module 4:** with model-ready features in hand, we build and evaluate the models themselves, where precision, recall, F1, and honest evaluation finally get the full treatment they have been promised.


---
# Try it yourself (optional)

**Exercise 1.** Engineer a `FamilySize` feature (`SibSp` plus `Parch` plus one for the passenger), add it to `numeric_features`, and re-run the cross-validated pipeline. Does the F1 score improve?

**Exercise 2.** Swap `StandardScaler` for `RobustScaler` in the numeric pipeline and compare cross-validated F1. Then explain why the difference is small, referring to what the Winsorizer already did.

**Exercise 3.** Write a function that takes a fitted pipeline and returns the number of features it produces, then use it to show how the output width changes when `Deck` is removed from `categorical_features`.


In [ ]:
# Exercise workspace





---
## Exercise solutions


In [ ]:
# Exercise 1: family size as an engineered feature
ex = model_df.copy()
ex["FamilySize"] = ex["SibSp"] + ex["Parch"] + 1

num2 = numeric_features + ["FamilySize"]
prep2 = ColumnTransformer([
    ("num", numeric_pipeline, num2),
    ("cat", categorical_pipeline, categorical_features),
    ("pass", "passthrough", passthrough_features)])
pipe2 = Pipeline([("preprocess", prep2),
                  ("model", LogisticRegression(max_iter=1000,
                                               random_state=RANDOM_STATE))])

X2 = ex[num2 + categorical_features + passthrough_features]
base_score = cross_val_score(pipe, X, y, cv=5, scoring="f1").mean()
new_score = cross_val_score(pipe2, X2, ex["Survived"], cv=5, scoring="f1").mean()
print("Without FamilySize:", round(base_score, 4))
print("With FamilySize:   ", round(new_score, 4))


In [ ]:
# Exercise 2: RobustScaler instead of StandardScaler
robust_numeric = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("winsorize", Winsorizer(lower_pct=0.01, upper_pct=0.99)),
    ("scale", RobustScaler())])

prep3 = ColumnTransformer([
    ("num", robust_numeric, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
    ("pass", "passthrough", passthrough_features)])
pipe3 = Pipeline([("preprocess", prep3),
                  ("model", LogisticRegression(max_iter=1000,
                                               random_state=RANDOM_STATE))])

print("StandardScaler:", round(base_score, 4))
print("RobustScaler:  ", round(cross_val_score(pipe3, X, y, cv=5,
                                               scoring="f1").mean(), 4))
print()
print("The difference is small because the Winsorizer already capped the")
print("extreme fares, so StandardScaler's mean and std were not being")
print("distorted by outliers in the first place. RobustScaler's advantage")
print("appears when outliers are still present at scaling time.")


In [ ]:
# Exercise 3: counting output features
def count_output_features(fitted_pipeline):
    step = fitted_pipeline.named_steps["preprocess"]
    return len(step.get_feature_names_out())

print("With Deck:   ", count_output_features(best_pipe), "features")

prep4 = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, ["Sex", "Embarked"]),
    ("pass", "passthrough", passthrough_features)])
pipe4 = Pipeline([("preprocess", prep4),
                  ("model", LogisticRegression(max_iter=1000,
                                               random_state=RANDOM_STATE))])
pipe4.fit(X_train, y_train)
print("Without Deck:", count_output_features(pipe4), "features")
